# Prepare NaPTAN transport-nodes for MSOAs

In this notebook the number of active public transport stop points is accumulated and assigned in a table to each Greater London MSOA. One active NaPTAN stop point will be accounted for as one transport-node and these will later be used for an indicator of the local accessibility to that MSOA.

In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [2]:
MSOA_PATH = Path("../data/spatial/processed/london_msoa_2021.gpkg")

NAPTAN_PATH = Path("../data/spatial/raw/national_stop_data.csv")

OUTPUT_PATH = Path("../data/spatial/processed/london_msoa_transport_nodes.csv")

# Loading required fields from dataset

Only the required NaPTAN columns were parsed and loaded so that only fields that are needed for validating, identifying and locating the stop points are used, reducing the memory that would be needed for the whole dataset. The MSOA dataset is loaded first and checked for the correct number of rows and columns being loaded as well as the coordinate reference system which will be needed later for the MSOA to transport stop allocation.

In [3]:
london_msoa = gpd.read_file(MSOA_PATH)

print(f"London MSOA rows: {len(london_msoa)}")
print(f"London MSOA CRS: {london_msoa.crs}")
print(f"London MSOA columns: {london_msoa.columns.tolist()}")


naptan_data = pd.read_csv(
    NAPTAN_PATH,
    usecols=[
        "ATCOCode",
        "Status",
        "Easting",
        "Northing",
        "StopType"],
    dtype={
        "ATCOCode": "string",
        "Status": "string",
        "StopType": "string"},
    low_memory=False)

print(f"\nNational NaPTAN rows: {len(naptan_data)}")
print(f"Loaded columns: {naptan_data.columns.tolist()}")

print(f"\nStatus counts: \n{naptan_data["Status"].value_counts(dropna=False)}")

London MSOA rows: 1002
London MSOA CRS: EPSG:27700
London MSOA columns: ['borough_name', 'borough_code', 'msoa_name', 'msoa_code', 'area_km2', 'geometry']

National NaPTAN rows: 435261
Loaded columns: ['ATCOCode', 'Easting', 'Northing', 'StopType', 'Status']

Status counts: 
Status
active      387817
inactive     47443
pending          1
Name: count, dtype: Int64


# Filter for active stop points

The NaPTAN dataset contains both active, inactive and pending stop points and as such must be filtered for only the active stop points. The active points are selected and the records without valid coordinate values are removed to get a dataset with active transport nodes and numeric coordinate values.

In [4]:
active_status_naptan = naptan_data.loc[naptan_data["Status"].str.lower().eq("active")].copy()

active_status_naptan["Easting"] = pd.to_numeric(active_status_naptan["Easting"], errors="coerce")
active_status_naptan["Northing"] = pd.to_numeric(active_status_naptan["Northing"], errors="coerce")

active_status_naptan = active_status_naptan.dropna(subset=["Easting", "Northing"])

active_status_naptan = active_status_naptan.loc[(active_status_naptan["Easting"] > 0)
                                  & (active_status_naptan["Northing"] > 0)].copy()

active_status_naptan = active_status_naptan[["ATCOCode", "StopType", "Easting", "Northing"]]

print(f"Active NaPTAN rows with usable coordinates: {len(active_status_naptan)}")
print(f"Duplicate active ATCO codes: {active_status_naptan["ATCOCode"].duplicated().sum()}")

Active NaPTAN rows with usable coordinates: 387817
Duplicate active ATCO codes: 0


# Filter for Greater London

Using the london MSOA boundaries that have been established previously, the active NaPTAN transport nodes can further be filtered to only the active Greater London NaPTAN transport nodes. Using these, the easting and northing coordinates are converted into point geometries in the boundaries and matched to the boundaries.

In [5]:
min_x, min_y, max_x, max_y = london_msoa.total_bounds

possible_greater_london_naptan_node = active_status_naptan.loc[active_status_naptan["Easting"].between(min_x, max_x)
                                                               & active_status_naptan["Northing"].between(min_y, max_y)].copy()

print(f"Stop points inside London bounding box: {len(possible_greater_london_naptan_node)}")

Stop points inside London bounding box: 27672


In [6]:
greater_london_naptan_nodes = gpd.GeoDataFrame(
    possible_greater_london_naptan_node,
    geometry=gpd.points_from_xy(possible_greater_london_naptan_node["Easting"], possible_greater_london_naptan_node["Northing"]),
    crs="EPSG:27700")

print(f"NaPTAN point rows: {len(greater_london_naptan_nodes)}")
print(f"NaPTAN point CRS: {greater_london_naptan_nodes.crs}")

NaPTAN point rows: 27672
NaPTAN point CRS: EPSG:27700


# Assigning active stop points to MSOA codes

A spatial join can now assign the NaPTAN points to the London MSOA polygon that it is found within and an inner join is used to exclude points outside Greater london that may still be within the bounding boxes creater earlier. The stop type distribution is added in case it allows for any further findings. 

In [7]:
msoa_for_node_join = london_msoa[["msoa_code", "geometry"]].copy()

greater_london_stop_points = gpd.sjoin(
    greater_london_naptan_nodes,
    msoa_for_node_join,
    how="inner",
    predicate="within")

greater_london_stop_points = greater_london_stop_points.drop(columns="index_right")

print(f"Active stop points assigned to London MSOAs: {len(greater_london_stop_points)}")

print(f"\nLondon stop types: {greater_london_stop_points["StopType"].value_counts()}")

Active stop points assigned to London MSOAs: 22821

London stop types: StopType
BCT    20014
PLT      692
TMU      641
RSE      549
RLY      366
MET      345
FBT       81
FER       45
FTD       42
BCS       28
BCQ       10
GAT        8
Name: count, dtype: Int64


# Sum transport nodes within each MSOA

The spatially assigned points can be grouped by MSOA code and represent the number of active NaPTAN transport nodes found in each MSOA. Any MSOA's with no transport nodes are also retained in the dataset so that MSOAs without transport nodes are also represented.

In [8]:
msoa_transport_counts = (greater_london_stop_points
                         .groupby("msoa_code")
                         .size()
                         .reset_index(name="transport_nodes"))

print(f"MSOAs containing at least one transport node: {len(msoa_transport_counts)}")

MSOAs containing at least one transport node: 1001


In [9]:
greater_london_msoa_transport = (
    london_msoa[["msoa_code"]]
    .merge(msoa_transport_counts, on="msoa_code", how="left")
    .sort_values("msoa_code")
    .reset_index(drop=True)
)

greater_london_msoa_transport["transport_nodes"] = (greater_london_msoa_transport["transport_nodes"].fillna(0).astype("int64"))

print(f"Final MSOA rows: {len(greater_london_msoa_transport)}")

greater_london_msoa_transport.head()

Final MSOA rows: 1002


,msoa_code,transport_nodes
0,E02000001,235
1,E02000002,20
2,E02000003,21
3,E02000004,13
4,E02000005,11


# Validating final transport-node by MSOA table

The check confirms that all 1002 Greater London MSOAs are represented in the final dataset and non-London codes or missing/duplicates counts haven't been created and aren't included. The sum of the MSOA-level counts should also match that of the number of stop points produced in the spatial join.

In [10]:
missing_msoa_codes = (set(london_msoa["msoa_code"])- set(greater_london_msoa_transport["msoa_code"]))

extra_msoa_codes = (set(greater_london_msoa_transport["msoa_code"])- set(london_msoa["msoa_code"]))

print(f"London MSOA data rows: {len(london_msoa)}")
print(f"Transport output rows: {len(greater_london_msoa_transport)}")

print(f"\nDuplicate MSOA codes: {greater_london_msoa_transport["msoa_code"].duplicated().sum()}")
print(f"Missing MSOA codes: {greater_london_msoa_transport["msoa_code"].isna().sum()}")
print(f"Missing transport counts: {greater_london_msoa_transport["transport_nodes"].isna().sum()}")
print(f"Negative transport counts: {(greater_london_msoa_transport["transport_nodes"] < 0).sum()}")

print(f"\nLondon MSOAs absent from output: {len(missing_msoa_codes)}")
print(f"Non-London MSOAs in output: {len(extra_msoa_codes)}")

print(f"\nTransport node counts agree after join: {greater_london_msoa_transport["transport_nodes"].sum()== len(greater_london_stop_points)}")
print(f"MSOAs with zero transport nodes: {(greater_london_msoa_transport["transport_nodes"] == 0).sum()}")

London MSOA data rows: 1002
Transport output rows: 1002

Duplicate MSOA codes: 0
Missing MSOA codes: 0
Missing transport counts: 0
Negative transport counts: 0

London MSOAs absent from output: 0
Non-London MSOAs in output: 0

Transport node counts agree after join: True
MSOAs with zero transport nodes: 1


# Save validated Greater London transport node table

In [11]:
greater_london_msoa_transport.to_csv(OUTPUT_PATH, index=False)

print(f"Saved to: {OUTPUT_PATH}")

Saved to: ..\data\spatial\processed\london_msoa_transport_nodes.csv
